# Agent 6 — Personal Memory Agent (LUCI)

> Index a user's daily recordings — wearable camera footage, phone
> videos, meeting recordings, screen captures — and answer natural-
> language questions about their past via search + transcript +
> VLM identification.

## What this notebook shows

- How to use the `datetime_taken` filter on `/search` for time-windowed
  queries ("last Tuesday's lunch").
- How to combine semantic search with exact-phrase transcript lookup
  for grounded answers.
- How to ask the VLM to read signage / context from a specific
  time window to identify locations and activities.

## Endpoints exercised

| Step | Endpoint |
|---|---|
| Retrieve (time-windowed) | `POST /search` with `datetime_taken` |
| Retrieve (exact phrase) | `GET /search_audio_transcripts` |
| Reason | `POST /vu/chat/completions` |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


In [ ]:
def search_transcripts(query, *, unique_id="default", page=1, page_size=100):
    """Visual Search — GET /search_audio_transcripts.

    Exact-phrase LIKE matching over stored audio transcripts. Use when you
    know roughly what was said. For semantic audio search, use /search with
    search_type="BY_AUDIO" instead.
    """
    r = requests.get(
        f"{VS_HOST}/search_audio_transcripts",
        headers=HEADERS,
        params={"query": query, "unique_id": unique_id, "page": page, "page_size": page_size},
        timeout=30,
    )
    r.raise_for_status()
    envelope = r.json()
    assert envelope.get("code") == "0000", envelope
    return envelope.get("data") or {}


In [ ]:
def vlm_complete(prompt, *, video_url=None, image_url=None, system=None,
                 model=VLM_MODEL, response_json=True, temperature=0.2,
                 max_tokens=1024):
    """Visual Intelligence — POST /vu/chat/completions.

    Calls a Video Language Model (Gemini by default) with text + an
    optional media reference. Returns the assistant's text reply, with
    markdown ```json fences stripped if `response_json=True`.

    Notes on the wire format:
      • `content` MUST be an array even for text-only prompts. A bare
        string is rejected with "Model input cannot be empty".
      • Gemini's response lives in choices[0]["text"]. Other providers
        (Qwen, Nova) use choices[0]["message"]["content"]. We accept both.
      • The endpoint can return HTTP 200 with status="errored" — surface
        that as a typed exception rather than letting JSON parsing fail.
    """
    content = [{"type": "text", "text": prompt}]
    if video_url:
        content.append({"type": "input_file", "file_uri": video_url, "mime_type": "video/mp4"})
    if image_url:
        content.append({"type": "input_file", "file_uri": image_url, "mime_type": "image/jpeg"})

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_json:
        # Tells Gemini to bias toward JSON output. Other providers ignore this.
        body["extra_body"] = {"metadata": {"response_mime_type": "application/json"}}

    r = requests.post(f"{VLM_HOST}/vu/chat/completions", headers=HEADERS,
                      json=body, timeout=180)
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("status") == "errored" or envelope.get("error"):
        err = envelope.get("error") or {}
        raise RuntimeError(f"VLM error: {err.get('code')} {err.get('message')}")
    choices = envelope.get("choices") or []
    if not choices:
        raise RuntimeError(f"VLM returned no choices: {envelope}")
    # Two shapes observed in the wild.
    text = choices[0].get("text") or (choices[0].get("message") or {}).get("content", "")
    if response_json:
        text = _strip_json_fence(text)
    return text


def _strip_json_fence(text):
    """Gemini often wraps JSON output in ```json ... ``` fences even when
    response_mime_type=application/json. Trim them so json.loads() works."""
    if not text:
        return text
    s = text.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s[3:]
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s


In [ ]:
def resolve_media_url(video_no):
    """Bridge a videoNo to a public URL the VLM can fetch.

    `/download` streams the raw bytes back to you — it does NOT return a
    hosted URL. To feed a video to /vu/chat/completions you must host it
    yourself. This helper expects either:

      • MEMORIES_MEDIA_URL_TEMPLATE env var (e.g. https://your-cdn/{video_no}.mp4)
      • a MEDIA_URL_MAP dict you populate inline

    If neither is configured, raises so the notebook stops cleanly.
    """
    if video_no in MEDIA_URL_MAP:
        return MEDIA_URL_MAP[video_no]
    tpl = os.environ.get("MEMORIES_MEDIA_URL_TEMPLATE")
    if tpl:
        return tpl.format(video_no=video_no)
    raise RuntimeError(
        f"No media-URL bridge configured for {video_no}. "
        "Set MEMORIES_MEDIA_URL_TEMPLATE or add an entry to MEDIA_URL_MAP."
    )

# Per-notebook overrides: populate this for testing without a CDN.
# A public Memories.ai test asset is included as an example.
MEDIA_URL_MAP = {
    # "VI676024023022092288": "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
}


## Step 1 — turn the user's question into a search query

The question is free text. The agent's first job is to extract a
searchable phrase and a time window. A real LUCI deployment would use
an LLM to do this slot-filling; for this notebook we hard-code it.


In [ ]:
QUESTION   = "What did I order for lunch last Tuesday?"
DATE_AFTER = "2026-05-05 11:00:00"   # ← derived from "last Tuesday" + lunch
UNIQUE_ID  = "default"

# Search query is paraphrased from the question — semantic search is
# forgiving about wording but specific concepts (lunch, restaurant)
# tend to retrieve better than vague ones.
SEARCH_QUERY = "ordering food at a restaurant for lunch"


## Step 2 — time-windowed semantic search

The `datetime_taken` filter restricts hits to videos captured at or
after the given timestamp. To bound the top end, do a client-side
filter — the API doesn't expose a `datetime_taken_before` parameter.


In [ ]:
hits = search(
    SEARCH_QUERY,
    unique_id=UNIQUE_ID,
    datetime_taken=DATE_AFTER,
    top_k=5,
    filtering_level="medium",
)

if not hits:
    print("No matching memory found in that window.")
else:
    top = hits[0]
    print(f"Top hit: {top['videoNo']} @ {top['startTime']}-{top['endTime']}s "
          f"score={top['score']:.3f}")


## Step 3 — pull a transcript clue for the matched moment

Spoken words near the visual moment often answer the question directly
("I'll have the grilled salmon..."). We search the user's transcripts
for a plausible polite-ordering prefix.


In [ ]:
transcript_hint = None
try:
    tx = search_transcripts("I'll have", unique_id=UNIQUE_ID, page_size=10)
    for v in tx.get("videos", []):
        if v.get("videoNo") == top["videoNo"]:
            transcript_hint = v.get("audio_ts")
            break
except Exception as e:
    print(f"(transcript lookup skipped: {e})")

print(f"transcript_hint: {transcript_hint!r}")


## Step 4 — VLM identifies the venue + the order

Send the matched moment to Gemini with a focused JSON-schema prompt.
The transcript hint is included if we have one — it speeds up
identification dramatically.


In [ ]:
MEDIA_URL_MAP[top["videoNo"]] = "https://storage.googleapis.com/memories-test-data/test_1min.mp4"
media_url = resolve_media_url(top["videoNo"])

SYSTEM = ("You are LUCI, a personal memory assistant. Combine visual "
          "evidence and any transcript fragment to identify the venue, "
          "activity, and order. Reply strict JSON only.")
extra = f' Transcript hint: "{transcript_hint}".' if transcript_hint else ""
prompt = (
    f"Between {top['startTime']}s and {top['endTime']}s in this recording, "
    "identify: (1) venue/restaurant name (read signage), (2) what the user "
    "is doing, (3) what is being ordered or eaten. "
    'Reply JSON: {"venue": str|null, "activity": str, "order": str|null}.' + extra
)

raw = vlm_complete(prompt, video_url=media_url, system=SYSTEM)
evidence = json.loads(raw) if raw else {}
print(json.dumps(evidence, indent=2))


## Step 5 — compose a natural-language answer

Glue the structured fields back into a sentence to answer the user's
original question.


In [ ]:
venue   = evidence.get("venue")
order   = evidence.get("order")

pieces = []
if venue:
    pieces.append(f"at {venue}")
if order:
    pieces.append(f"you ordered {order}")

answer = f"Based on the recording, {', '.join(pieces) or 'I found a related moment'}."
print(f"\nQuestion: {QUESTION}\nAnswer:   {answer}")


## Where to go next

- **Multi-recording aggregation**: "Am I exercising more this month
  than last?" — run the same flow over two time ranges and diff.
- **Quantitative queries**: "How many pushups did I do this week?"
  requires the VLM to count reps per clip and sum across clips. Use
  a tighter schema: `{"is_pushups": bool, "rep_count": int}`.
- **Cross-modal queries**: the same indexed video answers "what
  restaurant?" (visual) AND "what did I order?" (audio) AND "what
  song was playing?" (audio + VLM identification) — universal
  indexing means no re-indexing per question.
